In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Load your dataset
df = pd.read_csv("diabetes_risk.csv")

# 2. Clean/prepare data
if "patient_id" in df.columns:
  df = df.drop(columns=["patient_id"])

for col in ["smoking_status", "alcohol_consumption", "income_bracket"]:
  if col in df.columns:
    df[col] = df[col].fillna("Unknown")

risk_mapping = {"Low": 0, "Moderate": 1, "High": 2}
if df["diabetes_risk"].dtype == "object":
  df["diabetes_risk_encoded"] = df["diabetes_risk"].map(risk_mapping)
else:
  df["diabetes_risk_encoded"] = df["diabetes_risk"]

X = df.drop(columns=["diabetes_risk", "diabetes_risk_encoded"])
y = df["diabetes_risk_encoded"]

numerical_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

# 3. Build the pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_cols,
        ),
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", GradientBoostingClassifier(random_state=42)),
    ]
)

# 4. Train the model
pipeline.fit(X, y)

# 5. Save the trained pipeline to a .pkl file
joblib.dump(pipeline, "diabetes_risk_model.pkl")
print("✅ Success! True pipeline model trained and saved.")

✅ Success! True pipeline model trained and saved.
